In [ ]:
import os
import time
import random
import datetime
import pandas as pd
from dotenv import load_dotenv
from cassandra.cluster import Cluster
import cassandra.util
import mysql.connector

# Load environment variables
load_dotenv()

# MySQL Configuration
MYSQL_HOST = os.getenv('MYSQL_HOST', 'localhost')
MYSQL_PORT = os.getenv('MYSQL_PORT', '3306')
MYSQL_DB = os.getenv('MYSQL_DB', 'etl_db')
MYSQL_USER = os.getenv('MYSQL_USER', 'root')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD', '1')

# Cassandra Configuration
CASSANDRA_HOST = os.getenv('CASSANDRA_HOST', 'localhost')
CASSANDRA_KEYSPACE = os.getenv('CASSANDRA_KEYSPACE', 'recruitment')
CASSANDRA_TABLE = os.getenv('CASSANDRA_TABLE', 'tracking')

# Initialize Cassandra connection
print(f"Connecting to Cassandra at {CASSANDRA_HOST}...")
cluster = Cluster([CASSANDRA_HOST])
session = cluster.connect(CASSANDRA_KEYSPACE)

# Prepare the statement for efficiency and type safety
query = f"""
    INSERT INTO {CASSANDRA_TABLE}
    (create_time, bid, campaign_id, custom_track, group_id, job_id, publisher_id, ts)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
"""
prepared = session.prepare(query)

def get_data_from_job():
    cnx = mysql.connector.connect(
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        host=MYSQL_HOST, database=MYSQL_DB
    )
    query = "SELECT id as job_id, campaign_id, group_id, company_id FROM job"
    mysql_data = pd.read_sql(query, cnx)
    cnx.close()
    return mysql_data

def get_data_from_publisher():
    cnx = mysql.connector.connect(
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        host=MYSQL_HOST, database=MYSQL_DB
    )
    query = "SELECT DISTINCT(id) as publisher_id FROM master_publisher"
    mysql_data = pd.read_sql(query, cnx)
    cnx.close()
    return mysql_data

def generating_dummy_data(n_records, job_list, campaign_list, company_list, group_list, publisher_list):
    for _ in range(n_records):
        # Create TimeUUID object
        create_time = cassandra.util.uuid_from_time(datetime.datetime.now())

        # Values setup
        bid = str(random.randint(0, 1)) # Convert to string to match Cassandra 'text' type
        interact = ['click', 'conversion', 'qualified', 'unqualified']
        custom_track = random.choices(interact, weights=(70, 10, 10, 10))[0]
        job_id = random.choice(job_list)
        publisher_id = random.choice(publisher_list)
        group_id = random.choice(group_list)
        campaign_id = random.choice(campaign_list)
        ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        # Execute using prepared statement (handles quoting automatically)
        session.execute(prepared, (
            create_time,
            bid,
            int(campaign_id),
            custom_track,
            int(group_id),
            int(job_id),
            int(publisher_id),
            ts
        ))
    print(f"--- {n_records} Records Generated Successfully ---")

if __name__ == "__main__":
    print(f"Starting data generator...")

    # Pre-fetch dimension data to avoid overhead in the loop
    jobs_data = get_data_from_job()
    publisher = get_data_from_publisher()['publisher_id'].to_list()

    job_list = jobs_data['job_id'].to_list()
    campaign_list = jobs_data['campaign_id'].to_list()
    company_list = jobs_data['company_id'].to_list()
    group_list = jobs_data[jobs_data['group_id'].notnull()]['group_id'].astype(int).to_list()

    try:
        while True:
            n = random.randint(1, 20)
            generating_dummy_data(n, job_list, campaign_list, company_list, group_list, publisher)
            time.sleep(20)
    except KeyboardInterrupt:
        print("\nStopping data generator...")
    finally:
        cluster.shutdown()


Connecting to Cassandra at cassandra_etl...
Starting data generator...
--- 20 Records Generated Successfully ---


/tmp/ipykernel_32/3287231149.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mysql_data = pd.read_sql(query, cnx)
/tmp/ipykernel_32/3287231149.py:55: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mysql_data = pd.read_sql(query, cnx)


--- 10 Records Generated Successfully ---
--- 13 Records Generated Successfully ---
--- 8 Records Generated Successfully ---
--- 20 Records Generated Successfully ---
--- 4 Records Generated Successfully ---
--- 7 Records Generated Successfully ---
--- 12 Records Generated Successfully ---
--- 19 Records Generated Successfully ---
--- 1 Records Generated Successfully ---
--- 19 Records Generated Successfully ---
--- 16 Records Generated Successfully ---
--- 4 Records Generated Successfully ---
--- 7 Records Generated Successfully ---
--- 4 Records Generated Successfully ---
--- 5 Records Generated Successfully ---
--- 19 Records Generated Successfully ---
--- 5 Records Generated Successfully ---
--- 15 Records Generated Successfully ---
--- 12 Records Generated Successfully ---
--- 3 Records Generated Successfully ---
--- 1 Records Generated Successfully ---
--- 15 Records Generated Successfully ---
--- 4 Records Generated Successfully ---
--- 14 Records Generated Successfully ---
--- 